In [ ]:
import os
import re
import json
import time
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset

ZIP_PATH = "/content/SimulationCsvs.2721.zip"

DATASET_DIR = "/content/dataset_v8_cellsplit"
NN_OUT_DIR  = "/content/nn_results_v8_cellsplit"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(NN_OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_RESAMPLE = 1000

N_VAL_RUNS  = 8
N_TEST_RUNS = 16

STRESS_SCALE   = 55.0e6
LOGRHO_SHIFT   = 12.0
SDOT_LOG_SHIFT = 13.76

DP_HIDDEN_DIMS   = [128, 256, 128, 64]
DP_ACTIVATION    = "SiLU"
DP_DROPOUT       = 0.05
DP_USE_BATCHNORM = False

RHO_HIDDEN_DIMS   = [128, 128, 64, 32]
RHO_ACTIVATION    = "SiLU"
RHO_DROPOUT       = 0.05
RHO_USE_BATCHNORM = False

BATCH_SIZE       = 512
ROLL_BATCH_SIZE  = 128
MAX_EPOCHS       = 2500
LR_INIT          = 3e-4
T0               = 100
T_MULT           = 2
LR_MIN_COSINE    = 1e-6
ES_PATIENCE      = 250
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 5.0

INPUT_NOISE_STD  = 0.01
HUBER_DELTA      = 1.0

WINDOW_LEN          = 24
WINDOW_STRIDE       = 12
ROLL_START_EPOCH    = 1

W_STEP_DP           = 1.0
W_ROLL_DP           = 0.35

W_STEP_RHO          = 1.0
W_ROLL_RHO          = 0.50

MAX_WINDOWS_PER_RUN = None

LP_COLS  = ["Lpxx","Lpxy","Lpxz","Lpyx","Lpyy","Lpyz","Lpzx","Lpzy","Lpzz"]
SIG_COLS = ["s_xx(Pa)","s_yy(Pa)","s_zz(Pa)","s_yz(Pa)","s_zx(Pa)","s_xy(Pa)"]
EP_COL   = "ep_eq(-)"
RHO_COL  = "density(1/m^2)"

INPUT_LABELS_DP = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                   'σ_dot','log10ρ','loading','εp_cum']

INPUT_LABELS_RHO = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                    'σ_dot','log10ρ','loading','εp_cum','dt']
DP_OUTPUT_LABELS_5 = ['Dp_xx','Dp_yy','Dp_yz','Dp_xz','Dp_xy']
DP_OUTPUT_LABELS_6 = ['Dp_xx','Dp_yy','Dp_zz','Dp_yz','Dp_xz','Dp_xy']

TEST_CELL_SPECS = [
    (40, "4e13", "1.43e12"),
    (40, "6e13", "1.43e12"),
    (40, "8e13", "1.43e12"),
    (40, "6e13", "2.055e12"),
    (40, "6e13", "9.5668e11"),
    (30, "8e13", "2.055e12"),
    (50, "8e13", "9.5668e11"),
    (55, "8e13", "1.43e12"),
]

VAL_CELL_SPECS = [
    (30, "4e13", "1.43e12"),
    (40, "4e13", "9.5668e11"),
    (50, "6e13", "2.055e12"),
    (55, "6e13", "9.5668e11"),
]

CELL_SPLIT_GUARANTEES = {
    "orientation_pair": "Uniaxial_40Mpa_6e13_1.43e12 vs shear_40Mpa_6e13_1.43e12 (both held-out in TEST)",
    "rate_sweep_uniaxial": "40 MPa, rho0=1.43e12, rates {4e13,6e13,8e13} (all in TEST)",
    "rate_sweep_shear": "40 MPa, rho0=1.43e12, rates {4e13,6e13,8e13} (all in TEST)",
    "density_sweep_uniaxial": "40 MPa, rate=6e13, rho0={1.43e12,2.055e12,9.5668e11} (all in TEST)",
    "density_sweep_shear": "40 MPa, rate=6e13, rho0={1.43e12,2.055e12,9.5668e11} (all in TEST)",
    "balanced_target_coverage": "30/40/50/55 MPa each contribute a full uniaxial+shear pair to TEST",
}


def set_all_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dp6_to_epdot_eq_np(Dp6):
    T = np.array([
        [Dp6[0], Dp6[5], Dp6[4]],
        [Dp6[5], Dp6[1], Dp6[3]],
        [Dp6[4], Dp6[3], Dp6[2]],
    ], dtype=np.float64)
    return np.sqrt((2.0 / 3.0) * np.sum(T * T))


def dp5_to_epdot_eq_torch(dp5_phys):
    dxx = dp5_phys[..., 0]
    dyy = dp5_phys[..., 1]
    dzz = -(dxx + dyy)
    dyz = dp5_phys[..., 2]
    dxz = dp5_phys[..., 3]
    dxy = dp5_phys[..., 4]

    TT = dxx*dxx + dyy*dyy + dzz*dzz + 2.0*(dxy*dxy + dxz*dxz + dyz*dyz)
    epdot = torch.sqrt((2.0 / 3.0) * torch.clamp(TT, min=1e-40))
    return epdot


def reconstruct_Y6_from_Y5(Y5_phys):
    Dp_xx = Y5_phys[:, 0:1]
    Dp_yy = Y5_phys[:, 1:2]
    Dp_zz = -(Dp_xx + Dp_yy)
    Dp_yz = Y5_phys[:, 2:3]
    Dp_xz = Y5_phys[:, 3:4]
    Dp_xy = Y5_phys[:, 4:5]
    return np.hstack([Dp_xx, Dp_yy, Dp_zz, Dp_yz, Dp_xz, Dp_xy])


def parse_filename(name: str):

    stem = Path(name).stem
    loading = 0 if re.search(r'(?i)uniaxial', stem) else \
              1 if re.search(r'(?i)shear', stem) else -1

    nums = [float(x) for x in re.findall(
        r'[\d]+(?:\.[\d]+)?(?:e[+-]?[\d]+)?', stem, re.IGNORECASE
    )]
    target_mpa = sigma_dot = rho0 = None
    for n in nums:
        if 10 <= n <= 200 and target_mpa is None:
            target_mpa = n * 1e6
        elif 1e12 <= n <= 1e14 and sigma_dot is None:
            sigma_dot = n
        elif 1e10 <= n <= 1e13 and rho0 is None:
            rho0 = n

    cell_key = (target_mpa, sigma_dot, rho0)

    return dict(
        stem=stem,
        loading=loading,
        target_Pa=target_mpa,
        sigma_dot=sigma_dot,
        rho0=rho0,
        cell_key=cell_key,
        strat_key=f"load{loading}_rho{rho0:.2e}" if rho0 is not None else "unknown",
    )


def resample_run(df, meta, n_points):

    t = df["time(s)"].values.astype(float)
    required = SIG_COLS + LP_COLS + [EP_COL, RHO_COL]
    if len(np.unique(t)) < 2:
        return None
    if not all(c in df.columns for c in required):
        return None
    if meta["sigma_dot"] is None or meta["rho0"] is None:
        return None
    t_new = np.linspace(t[0], t[-1], n_points)
    dt_uniform = t_new[1] - t_new[0] if len(t_new) > 1 else 0.0

    def interp(cols):
        arr = df[cols].values.astype(float)
        out = np.zeros((n_points, len(cols)), float)
        for j in range(arr.shape[1]):
            f = interp1d(
                t, arr[:, j],
                kind="linear",
                bounds_error=False,
                fill_value=(arr[0, j], arr[-1, j]),
            )
            out[:, j] = f(t_new)
        return out

    sig    = interp(SIG_COLS)
    ep_cum = interp([EP_COL])

    rho = interp([RHO_COL]).reshape(-1)
    rho = np.maximum(rho, 1e6)
    log10rho = np.log10(rho)

    dlog10rho = np.zeros(n_points, dtype=float)
    dlog10rho[:-1] = log10rho[1:] - log10rho[:-1]
    dlog10rho[-1]  = dlog10rho[-2] if n_points > 1 else 0.0

    if dt_uniform > 0:
        dlog10rho_rate = dlog10rho / dt_uniform
    else:
        dlog10rho_rate = np.zeros_like(dlog10rho)

    Lp_raw = interp(LP_COLS)
    Lp_mat = Lp_raw.reshape(-1, 3, 3)

    Dp_mat = 0.5 * (Lp_mat + Lp_mat.transpose(0, 2, 1))
    Dp = np.stack([
        Dp_mat[:, 0, 0],
        Dp_mat[:, 1, 1],
        Dp_mat[:, 2, 2],
        Dp_mat[:, 1, 2],
        Dp_mat[:, 0, 2],
        Dp_mat[:, 0, 1],
    ], axis=1)
    return dict(
        sig            = sig,
        ep_cum         = ep_cum,
        rho            = rho,
        log10rho       = log10rho,
        dlog10rho      = dlog10rho,
        dlog10rho_rate = dlog10rho_rate,
        Dp             = Dp,
        sigma_dot      = np.full(n_points, meta["sigma_dot"], float),
        rho0           = np.full(n_points, meta["rho0"], float),
        loading        = np.full(n_points, float(meta["loading"]), float),
        time           = t_new,
        dt             = np.full(n_points, dt_uniform, float),
        stem           = meta["stem"],
        n_orig         = len(df),
        cell_key       = meta["cell_key"],
        strat_key      = meta["strat_key"],
    )


def _find_cell(cell_to_idxs, target_mpa, rate_str, rho0_str):
    rate_map = {"4e13": 4e13, "6e13": 6e13, "8e13": 8e13}
    rho0_map = {"1.43e12": 1.43e12, "2.055e12": 2.055e12, "9.5668e11": 9.5668e11}
    target_pa = target_mpa * 1e6
    rate_val = rate_map[rate_str]
    rho0_val = rho0_map[rho0_str]
    for key in cell_to_idxs:
        t, r, rho = key
        if (t is not None and abs(t - target_pa) < 1.0 and
            r is not None and abs(r - rate_val) / rate_val < 1e-6 and
            rho is not None and abs(rho - rho0_val) / rho0_val < 1e-3):
            return key
    return None


def cell_grouped_split(run_data, test_cell_specs=TEST_CELL_SPECS,
                       val_cell_specs=VAL_CELL_SPECS, seed=42):
    cell_to_idxs = defaultdict(list)
    for i, rd in enumerate(run_data):
        cell_to_idxs[rd["cell_key"]].append(i)

    test_cell_keys = [_find_cell(cell_to_idxs, *spec) for spec in test_cell_specs]
    test_cell_keys = [k for k in test_cell_keys if k is not None]
    if len(test_cell_keys) != len(test_cell_specs):
        print(f"  WARNING: {len(test_cell_specs) - len(test_cell_keys)} test cell spec(s) "
              f"not found in data: check TEST_CELL_SPECS against available conditions.")

    val_cell_keys = [_find_cell(cell_to_idxs, *spec) for spec in val_cell_specs]
    val_cell_keys = [k for k in val_cell_keys if k is not None]
    val_cell_keys = [k for k in val_cell_keys if k not in test_cell_keys]

    idx_test = []
    for key in test_cell_keys:
        idx_test.extend(cell_to_idxs[key])

    idx_val = []
    for key in val_cell_keys:
        idx_val.extend(cell_to_idxs[key])

    used = set(idx_test) | set(idx_val)
    idx_train = [i for i in range(len(run_data)) if i not in used]

    val_counts = {str(k): 1 for k in val_cell_keys}
    test_counts = {str(k): 1 for k in test_cell_keys}

    return idx_train, idx_val, idx_test, val_counts, test_counts


def assemble(run_data, idxs):
    sigs_dp, sdots_dp, logrhos_dp, loads_dp, eps_dp = [], [], [], [], []
    Y_dp = []

    sigs_rho, sdots_rho, logrhos_rho, loads_rho, eps_rho, dt_rho = [], [], [], [], [], []
    Y_rho = []

    for i in idxs:
        rd = run_data[i]

        sigs_dp.append(rd["sig"])
        sdots_dp.append(rd["sigma_dot"][:, None])
        logrhos_dp.append(rd["log10rho"][:, None])
        loads_dp.append(rd["loading"][:, None])
        eps_dp.append(rd["ep_cum"])

        Y_dp.append(np.column_stack([
            rd["Dp"][:, 0],
            rd["Dp"][:, 1],
            rd["Dp"][:, 3],
            rd["Dp"][:, 4],
            rd["Dp"][:, 5],
        ]))

        sigs_rho.append(rd["sig"])
        sdots_rho.append(rd["sigma_dot"][:, None])
        logrhos_rho.append(rd["log10rho"][:, None])
        loads_rho.append(rd["loading"][:, None])
        eps_rho.append(rd["ep_cum"])
        dt_rho.append(rd["dt"][:, None])

        Y_rho.append(rd["dlog10rho_rate"][:, None])

    X_dp = np.hstack([
        np.vstack(sigs_dp),
        np.vstack(sdots_dp),
        np.vstack(logrhos_dp),
        np.vstack(loads_dp),
        np.vstack(eps_dp),
    ])

    X_rho = np.hstack([
        np.vstack(sigs_rho),
        np.vstack(sdots_rho),
        np.vstack(logrhos_rho),
        np.vstack(loads_rho),
        np.vstack(eps_rho),
        np.vstack(dt_rho),
    ])

    Y_dp  = np.vstack(Y_dp)
    Y_rho = np.vstack(Y_rho)

    return X_dp.astype(np.float32), Y_dp.astype(np.float32), X_rho.astype(np.float32), Y_rho.astype(np.float32)


def save_run_sequences(run_data, idx_train, idx_val, idx_test, out_dir):
    split_map = {}
    for i in idx_train:
        split_map[run_data[i]["stem"]] = "train"
    for i in idx_val:
        split_map[run_data[i]["stem"]] = "val"
    for i in idx_test:
        split_map[run_data[i]["stem"]] = "test"

    rows = []
    for rd in run_data:
        stem = rd["stem"]
        split = split_map[stem]
        n = rd["sig"].shape[0]

        for k in range(n):
            rows.append({
                "stem": stem,
                "split": split,
                "k": k,
                "time": rd["time"][k],
                "dt": rd["dt"][k],
                "s_xx": rd["sig"][k, 0],
                "s_yy": rd["sig"][k, 1],
                "s_zz": rd["sig"][k, 2],
                "s_yz": rd["sig"][k, 3],
                "s_zx": rd["sig"][k, 4],
                "s_xy": rd["sig"][k, 5],
                "sigma_dot": rd["sigma_dot"][k],
                "rho0": rd["rho0"][k],
                "rho_true": rd["rho"][k],
                "log10rho_true": rd["log10rho"][k],
                "dlog10rho_true": rd["dlog10rho"][k],
                "dlog10rho_rate_true": rd["dlog10rho_rate"][k],
                "loading": rd["loading"][k],
                "ep_eq_cum": rd["ep_cum"][k, 0],
                "Dp_xx": rd["Dp"][k, 0],
                "Dp_yy": rd["Dp"][k, 1],
                "Dp_zz": rd["Dp"][k, 2],
                "Dp_yz": rd["Dp"][k, 3],
                "Dp_xz": rd["Dp"][k, 4],
                "Dp_xy": rd["Dp"][k, 5],
            })

    df_seq = pd.DataFrame(rows)
    seq_path = os.path.join(out_dir, "run_sequences_v8_cellsplit.csv")
    df_seq.to_csv(seq_path, index=False)
    print(f"Saved run sequences → {seq_path}")
    return seq_path


def build_dataset_v8():

    print(f"Reading {ZIP_PATH} ...")
    run_data = []

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        csv_names = sorted([
            n for n in zf.namelist()
            if n.lower().endswith(".csv") and not n.startswith("__")
        ])
        print(f"Found {len(csv_names)} CSV files")

        for name in csv_names:
            meta = parse_filename(name)
            try:
                with zf.open(name) as f:
                    df = pd.read_csv(f, comment="#")
                df.columns = [c.strip() for c in df.columns]

                rd = resample_run(df, meta, N_RESAMPLE)
                if rd is not None:
                    run_data.append(rd)
                    print(f"  ✓ {meta['stem'][:52]:52s} orig={rd['n_orig']:6d} cell={rd['cell_key']}")
            except Exception as e:
                print(f"  ✗ {meta['stem']}: {e}")

    print(f"\nTotal runs: {len(run_data)}")

    idx_train, idx_val, idx_test, val_counts, test_counts = cell_grouped_split(
        run_data, test_cell_specs=TEST_CELL_SPECS, val_cell_specs=VAL_CELL_SPECS, seed=RANDOM_SEED
    )

    print(f"\nCell-grouped split: train={len(idx_train)}  val={len(idx_val)}  test={len(idx_test)} runs")
    print(f"  (previous scheme gave 52/8/8; new scheme trades some train data for")
    print(f"   guaranteed held-out comparison sets -- see guarantees below)")

    print("\nGuarantees satisfied by this test split (all within held-out data):")
    for k, v in CELL_SPLIT_GUARANTEES.items():
        print(f"  - {k}: {v}")

    print("\nTest set stems:")
    for i in sorted(idx_test, key=lambda i: run_data[i]["stem"]):
        print(f"  {run_data[i]['stem']}")

    print("\nVal set stems:")
    for i in sorted(idx_val, key=lambda i: run_data[i]["stem"]):
        print(f"  {run_data[i]['stem']}")

    assert len(set(idx_train) & set(idx_val)) == 0, "train/val overlap detected!"
    assert len(set(idx_train) & set(idx_test)) == 0, "train/test overlap detected!"
    assert len(set(idx_val) & set(idx_test)) == 0, "val/test overlap detected!"
    assert len(idx_train) + len(idx_val) + len(idx_test) == len(run_data), "split doesn't cover all runs!"
    print(f"\nSanity check passed: {len(idx_train)}+{len(idx_val)}+{len(idx_test)}={len(run_data)} runs, no overlap")

    X_train_dp_raw, Y_train_dp_raw, X_train_rho_raw, Y_train_rho_raw = assemble(run_data, idx_train)
    X_val_dp_raw,   Y_val_dp_raw,   X_val_rho_raw,   Y_val_rho_raw   = assemble(run_data, idx_val)
    X_test_dp_raw,  Y_test_dp_raw,  X_test_rho_raw,  Y_test_rho_raw  = assemble(run_data, idx_test)

    print(f"\nShapes:")
    print(f"  X_train_dp ={X_train_dp_raw.shape}  Y_train_dp ={Y_train_dp_raw.shape}")
    print(f"  X_val_dp   ={X_val_dp_raw.shape}    Y_val_dp   ={Y_val_dp_raw.shape}")
    print(f"  X_test_dp  ={X_test_dp_raw.shape}   Y_test_dp  ={Y_test_dp_raw.shape}")
    print(f"  X_train_rho={X_train_rho_raw.shape}  Y_train_rho={Y_train_rho_raw.shape}")
    print(f"  X_val_rho  ={X_val_rho_raw.shape}    Y_val_rho  ={Y_val_rho_raw.shape}")
    print(f"  X_test_rho ={X_test_rho_raw.shape}   Y_test_rho ={Y_test_rho_raw.shape}")

    Y_dp_std = Y_train_dp_raw.std(axis=0).astype(np.float64)
    Y_dp_std[Y_dp_std < 1e-40] = 1.0

    Y_rho_std = float(Y_train_rho_raw.std())
    if Y_rho_std < 1e-40:
        Y_rho_std = 1.0

    EP_MAX = float(X_train_dp_raw[:, 9].max())

    dt_log_train = np.log10(np.maximum(X_train_rho_raw[:, 10], 1e-40))
    DT_LOG_MEAN = float(dt_log_train.mean())
    DT_LOG_STD  = float(dt_log_train.std())
    if DT_LOG_STD < 1e-12:
        DT_LOG_STD = 1.0
    stats = {
        "Y_dp_std": Y_dp_std.tolist(),
        "Y_rho_std": Y_rho_std,
        "EP_MAX": EP_MAX,
        "DT_LOG_MEAN": DT_LOG_MEAN,
        "DT_LOG_STD": DT_LOG_STD,
        "n_inputs_dp": 10,
        "n_inputs_rho": 11,
        "N_RESAMPLE": N_RESAMPLE,
        "input_order_dp": ["s_xx","s_yy","s_zz","s_yz","s_zx","s_xy",
                           "sigma_dot","log10rho","loading","ep_eq_cum"],
        "input_order_rho": ["s_xx","s_yy","s_zz","s_yz","s_zx","s_xy",
                            "sigma_dot","log10rho","loading","ep_eq_cum","dt"],
        "dp_output_order": ["Dp_xx","Dp_yy","Dp_yz","Dp_xz","Dp_xy"],
        "rho_output_order": ["dlog10rho_rate"],
        "window_len": WINDOW_LEN,
        "window_stride": WINDOW_STRIDE,
        "split_info": {
            "train_runs": len(idx_train),
            "val_runs": len(idx_val),
            "test_runs": len(idx_test),
            "split_scheme": "cell_grouped_split_v8",
            "guarantees": CELL_SPLIT_GUARANTEES,
        },
    }

    npz_path   = os.path.join(DATASET_DIR, "ddd_dataset_v8_cellsplit.npz")
    stats_path = os.path.join(DATASET_DIR, "norm_stats_v8_cellsplit.json")
    split_path = os.path.join(DATASET_DIR, "run_split_v8_cellsplit.csv")

    np.savez_compressed(
        npz_path,
        X_train_dp_raw=X_train_dp_raw,
        X_val_dp_raw=X_val_dp_raw,
        X_test_dp_raw=X_test_dp_raw,
        Y_train_dp_raw=Y_train_dp_raw,
        Y_val_dp_raw=Y_val_dp_raw,
        Y_test_dp_raw=Y_test_dp_raw,
        X_train_rho_raw=X_train_rho_raw,
        X_val_rho_raw=X_val_rho_raw,
        X_test_rho_raw=X_test_rho_raw,
        Y_train_rho_raw=Y_train_rho_raw,
        Y_val_rho_raw=Y_val_rho_raw,
        Y_test_rho_raw=Y_test_rho_raw,
    )

    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    rows = []
    for label, idxs in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
        for i in idxs:
            rows.append({
                "stem": run_data[i]["stem"],
                "split": label,
                "cell_key": str(run_data[i]["cell_key"]),
            })
    pd.DataFrame(rows).to_csv(split_path, index=False)

    seq_path = save_run_sequences(run_data, idx_train, idx_val, idx_test, DATASET_DIR)

    print(f"\nSaved:")
    print(f"  {npz_path}")
    print(f"  {stats_path}")
    print(f"  {split_path}")
    print(f"  {seq_path}")

    return {
        "npz_path": npz_path,
        "stats_path": stats_path,
        "seq_path": seq_path
    }


def normalize_X_dp(X_raw, EP_MAX):
    X = X_raw.copy().astype(np.float64)
    X[:, :6] = X[:, :6] / STRESS_SCALE
    X[:,  6] = np.log10(np.abs(X[:, 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[:,  7] = X[:,  7] - LOGRHO_SHIFT
    X[:,  8] = 2.0 * X[:, 8] - 1.0
    X[:,  9] = X[:,  9] / (EP_MAX + 1e-40)
    return X.astype(np.float32)


def normalize_X_rho(X_raw, EP_MAX, dt_log_mean, dt_log_std):
    X = X_raw.copy().astype(np.float64)
    X[:, :6] = X[:, :6] / STRESS_SCALE
    X[:,  6] = np.log10(np.abs(X[:, 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[:,  7] = X[:,  7] - LOGRHO_SHIFT
    X[:,  8] = 2.0 * X[:, 8] - 1.0
    X[:,  9] = X[:,  9] / (EP_MAX + 1e-40)

    dt_log = np.log10(np.maximum(X[:, 10], 1e-40))
    X[:, 10] = (dt_log - dt_log_mean) / (dt_log_std + 1e-40)
    return X.astype(np.float32)


def normalize_Y_dp(Y_raw, Y_std):
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_dp(Y_norm, Y_std):
    return Y_norm * Y_std


def normalize_Y_rho(Y_raw, Y_std):
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_rho(Y_norm, Y_std):
    return Y_norm * Y_std


class MLPRegressor(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dims, activation="SiLU",
                 dropout=0.0, use_bn=False):
        super().__init__()

        act_fn = getattr(nn, activation)
        layers = []
        prev = in_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act_fn())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class WeightedHuberLoss(nn.Module):
    def __init__(self, Y_std, delta=1.0):
        super().__init__()
        Y_std = np.asarray(Y_std, dtype=np.float32).reshape(-1)
        w = 1.0 / (Y_std + 1e-40)
        w = w / w.sum() * len(Y_std)
        self.register_buffer("weights", torch.tensor(w, dtype=torch.float32))
        self.huber = nn.HuberLoss(reduction="none", delta=delta)

    def forward(self, pred, target):
        return (self.huber(pred, target) * self.weights).mean()


class ScalarHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.huber = nn.HuberLoss(reduction="mean", delta=delta)

    def forward(self, pred, target):
        return self.huber(pred, target)


class NoisyDataset(Dataset):
    def __init__(self, X, Y, noise_std=0.01, no_noise_indices=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
        self.noise_std = noise_std
        self.no_noise_indices = [] if no_noise_indices is None else no_noise_indices

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.Y[idx]

        if self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            for j in self.no_noise_indices:
                noise[j] = 0.0
            x = x + noise

        return x, y


class DpRolloutWindowDataset(Dataset):

    def __init__(self, seq_csv_path, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        self.windows = []

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0
            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                item = {
                    "X_path_raw": np.column_stack([
                        sub["s_xx"].values,
                        sub["s_yy"].values,
                        sub["s_zz"].values,
                        sub["s_yz"].values,
                        sub["s_zx"].values,
                        sub["s_xy"].values,
                        sub["sigma_dot"].values,
                        sub["log10rho_true"].values,
                        sub["loading"].values,
                        sub["ep_eq_cum"].values,
                    ]).astype(np.float32),
                    "dt": sub["dt"].values.astype(np.float32),
                    "ep_true": sub["ep_eq_cum"].values.astype(np.float32),
                    "Dp_true_5": np.column_stack([
                        sub["Dp_xx"].values,
                        sub["Dp_yy"].values,
                        sub["Dp_yz"].values,
                        sub["Dp_xz"].values,
                        sub["Dp_xy"].values,
                    ]).astype(np.float32),
                }

                self.windows.append(item)
                count_for_run += 1

                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["X_path_raw"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["ep_true"], dtype=torch.float32),
            torch.tensor(item["Dp_true_5"], dtype=torch.float32),
        )


class RhoRolloutWindowDataset(Dataset):

    def __init__(self, seq_csv_path, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        self.windows = []

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0

            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                item = {
                    "X_path_raw": np.column_stack([
                        sub["s_xx"].values,
                        sub["s_yy"].values,
                        sub["s_zz"].values,
                        sub["s_yz"].values,
                        sub["s_zx"].values,
                        sub["s_xy"].values,
                        sub["sigma_dot"].values,
                        sub["log10rho_true"].values,
                        sub["loading"].values,
                        sub["ep_eq_cum"].values,
                        sub["dt"].values,
                    ]).astype(np.float32),
                    "dt": sub["dt"].values.astype(np.float32),
                    "log10rho_true": sub["log10rho_true"].values.astype(np.float32),
                    "rho_rate_true": sub["dlog10rho_rate_true"].values.astype(np.float32),
                }

                self.windows.append(item)
                count_for_run += 1

                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["X_path_raw"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["log10rho_true"], dtype=torch.float32),
            torch.tensor(item["rho_rate_true"], dtype=torch.float32),
        )


def make_val_loader(X, Y, batch_size):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(Y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)


def run_epoch_train_one_step(model, loader, loss_fn, optimizer, device):
    model.train()
    total = 0.0
    n = 0

    for Xb, Yb in loader:
        Xb, Yb = Xb.to(device), Yb.to(device)
        pred = model(Xb)
        loss = loss_fn(pred, Yb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total += loss.item()
        n += 1

    return total / max(n, 1)


def run_epoch_eval_one_step(model, loader, loss_fn, device):
    model.eval()
    total = 0.0
    n = 0

    with torch.no_grad():
        for Xb, Yb in loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            pred = model(Xb)
            total += loss_fn(pred, Yb).item()
            n += 1

    return total / max(n, 1)


def rollout_window_losses_dp(model, batch_X_path_raw, batch_dt, batch_ep_true,
                             batch_Dp_true_5, EP_MAX, Y_dp_std, device):
    B, T, _ = batch_X_path_raw.shape

    X_path_raw = batch_X_path_raw.to(device)
    dt = batch_dt.to(device)
    ep_true = batch_ep_true.to(device)
    Dp_true_5 = batch_Dp_true_5.to(device)

    Y_dp_std_t = torch.tensor(Y_dp_std, dtype=torch.float32, device=device)
    X_flat = X_path_raw.reshape(B*T, -1)
    X_flat_np = X_flat.detach().cpu().numpy()
    X_flat_norm = normalize_X_dp(X_flat_np, EP_MAX)
    X_flat_norm = torch.tensor(X_flat_norm, dtype=torch.float32, device=device)

    pred5_norm_flat = model(X_flat_norm)
    pred5_phys_flat = pred5_norm_flat * Y_dp_std_t[None, :]

    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        pred5_phys_flat, Dp_true_5.reshape(B*T, 5)
    )
    ep_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    ep_pred[:, 0] = ep_true[:, 0]

    for t in range(T):
        x_t_raw = X_path_raw[:, t, :].clone()
        x_t_raw[:, 9] = ep_pred[:, t]

        x_t_np = x_t_raw.detach().cpu().numpy()
        x_t_norm = normalize_X_dp(x_t_np, EP_MAX)
        x_t_norm = torch.tensor(x_t_norm, dtype=torch.float32, device=device)

        pred5_norm_t = model(x_t_norm)
        pred5_phys_t = pred5_norm_t * Y_dp_std_t[None, :]
        epdot_t = dp5_to_epdot_eq_torch(pred5_phys_t)
        if t + 1 < T:
            ep_pred[:, t + 1] = ep_pred[:, t] + epdot_t * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(ep_pred, ep_true)

    return step_loss, roll_loss


def run_epoch_train_dp(model, step_loader, roll_loader, optimizer, device, EP_MAX, Y_dp_std):
    model.train()
    step_loss_fn = WeightedHuberLoss(Y_dp_std, delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(roll_loader)

    for Xb, Yb in step_loader:
        Xb, Yb = Xb.to(device), Yb.to(device)

        pred_step = model(Xb)
        loss_step_main = step_loss_fn(pred_step, Yb)

        try:
            batch_roll = next(roll_iter)
        except StopIteration:
            roll_iter = iter(roll_loader)
            batch_roll = next(roll_iter)

        X_path_raw, dt, ep_true, Dp_true_5 = batch_roll
        loss_step_roll, loss_roll = rollout_window_losses_dp(
            model=model,
            batch_X_path_raw=X_path_raw,
            batch_dt=dt,
            batch_ep_true=ep_true,
            batch_Dp_true_5=Dp_true_5,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
            device=device,
        )
        combined = W_STEP_DP * loss_step_main + 0.20 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)

        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_combined += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def run_epoch_eval_dp(model, val_step_loader, val_roll_loader, device, EP_MAX, Y_dp_std):
    model.eval()
    step_loss_fn = WeightedHuberLoss(Y_dp_std, delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(val_roll_loader)

    with torch.no_grad():
        for Xb, Yb in val_step_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)

            pred_step = model(Xb)
            loss_step_main = step_loss_fn(pred_step, Yb)

            try:
                batch_roll = next(roll_iter)
            except StopIteration:
                roll_iter = iter(val_roll_loader)
                batch_roll = next(roll_iter)

            X_path_raw, dt, ep_true, Dp_true_5 = batch_roll
            loss_step_roll, loss_roll = rollout_window_losses_dp(
                model=model,
                batch_X_path_raw=X_path_raw,
                batch_dt=dt,
                batch_ep_true=ep_true,
                batch_Dp_true_5=Dp_true_5,
                EP_MAX=EP_MAX,
                Y_dp_std=Y_dp_std,
                device=device,
            )

            combined = W_STEP_DP * loss_step_main + 0.20 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)

            total_combined += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def train_dp_model(model, step_train_loader, step_val_loader,
                   roll_train_loader, roll_val_loader,
                   EP_MAX, Y_dp_std, out_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=T_MULT, eta_min=LR_MIN_COSINE
    )

    print(f"\nTraining DpNet_v8 (cell-split) up to {MAX_EPOCHS} epochs ...")

    history = dict(
        train=[], val=[], lr=[], best_epoch=0,
        train_step=[], val_step=[],
        train_roll=[], val_roll=[],
    )

    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_dp(
            model=model,
            step_loader=step_train_loader,
            roll_loader=roll_train_loader,
            optimizer=optimizer,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
        )

        va, va_step, va_roll = run_epoch_eval_dp(
            model=model,
            val_step_loader=step_val_loader,
            val_roll_loader=roll_val_loader,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
        )

        scheduler.step(epoch - 1)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["lr"].append(lr)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 50 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:5d} | train={tr:.5f} val={va:.5f} "
                f"| step(tr/va)=({tr_step:.5f}/{va_step:.5f}) "
                f"| roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) "
                f"| best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop for DpNet_v8 at epoch {epoch}  (best val={best_val:.5f} @ epoch {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    print(f"\nRestored best DpNet_v8 model from epoch {history['best_epoch']}")
    torch.save(best_state, out_path)
    print(f"Saved DpNet_v8 weights → {out_path}")

    return model, history, best_val


def rho_rollout_window_losses(model, batch_X_path_raw, batch_dt, batch_log10rho_true,
                              batch_rho_rate_true, EP_MAX, Y_rho_std,
                              dt_log_mean, dt_log_std, device):

    B, T, _ = batch_X_path_raw.shape

    X_path_raw = batch_X_path_raw.to(device)
    dt = batch_dt.to(device)
    log10rho_true = batch_log10rho_true.to(device)
    rho_rate_true = batch_rho_rate_true.to(device)

    y_std_t = torch.tensor(Y_rho_std, dtype=torch.float32, device=device)

    X_flat = X_path_raw.reshape(B*T, -1)
    X_flat_np = X_flat.detach().cpu().numpy()
    X_flat_norm = normalize_X_rho(X_flat_np, EP_MAX, dt_log_mean, dt_log_std)
    X_flat_norm = torch.tensor(X_flat_norm, dtype=torch.float32, device=device)

    pred_rate_norm_flat = model(X_flat_norm).reshape(B, T)
    pred_rate_phys_flat = pred_rate_norm_flat * y_std_t

    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        pred_rate_phys_flat, rho_rate_true
    )

    log10rho_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    log10rho_pred[:, 0] = log10rho_true[:, 0]

    for t in range(T):
        x_t_raw = X_path_raw[:, t, :].clone()
        x_t_raw[:, 7] = log10rho_pred[:, t]

        x_t_np = x_t_raw.detach().cpu().numpy()
        x_t_norm = normalize_X_rho(x_t_np, EP_MAX, dt_log_mean, dt_log_std)
        x_t_norm = torch.tensor(x_t_norm, dtype=torch.float32, device=device)

        pred_rate_norm_t = model(x_t_norm).squeeze(-1)
        pred_rate_phys_t = pred_rate_norm_t * y_std_t

        if t + 1 < T:
            log10rho_pred[:, t + 1] = log10rho_pred[:, t] + pred_rate_phys_t * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        log10rho_pred, log10rho_true
    )

    return step_loss, roll_loss


def run_epoch_train_rho(model, step_loader, roll_loader, optimizer, device,
                        EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    model.train()
    step_loss_fn = ScalarHuberLoss(delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(roll_loader)

    for Xb, Yb in step_loader:
        Xb, Yb = Xb.to(device), Yb.to(device)

        pred_step = model(Xb)
        loss_step_main = step_loss_fn(pred_step, Yb)

        try:
            batch_roll = next(roll_iter)
        except StopIteration:
            roll_iter = iter(roll_loader)
            batch_roll = next(roll_iter)

        X_path_raw, dt, log10rho_true, rho_rate_true = batch_roll
        loss_step_roll, loss_roll = rho_rollout_window_losses(
            model=model,
            batch_X_path_raw=X_path_raw,
            batch_dt=dt,
            batch_log10rho_true=log10rho_true,
            batch_rho_rate_true=rho_rate_true,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
            device=device,
        )

        combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)

        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_combined += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def run_epoch_eval_rho(model, val_step_loader, val_roll_loader, device,
                       EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    model.eval()
    step_loss_fn = ScalarHuberLoss(delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(val_roll_loader)

    with torch.no_grad():
        for Xb, Yb in val_step_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)

            pred_step = model(Xb)
            loss_step_main = step_loss_fn(pred_step, Yb)

            try:
                batch_roll = next(roll_iter)
            except StopIteration:
                roll_iter = iter(val_roll_loader)
                batch_roll = next(roll_iter)

            X_path_raw, dt, log10rho_true, rho_rate_true = batch_roll
            loss_step_roll, loss_roll = rho_rollout_window_losses(
                model=model,
                batch_X_path_raw=X_path_raw,
                batch_dt=dt,
                batch_log10rho_true=log10rho_true,
                batch_rho_rate_true=rho_rate_true,
                EP_MAX=EP_MAX,
                Y_rho_std=Y_rho_std,
                dt_log_mean=dt_log_mean,
                dt_log_std=dt_log_std,
                device=device,
            )

            combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)

            total_combined += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def train_rho_model(model, step_train_loader, step_val_loader,
                    roll_train_loader, roll_val_loader,
                    EP_MAX, Y_rho_std, dt_log_mean, dt_log_std, out_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=T_MULT, eta_min=LR_MIN_COSINE
    )

    print(f"\nTraining RhoNet_v8 (cell-split) up to {MAX_EPOCHS} epochs ...")

    history = dict(
        train=[], val=[], lr=[], best_epoch=0,
        train_step=[], val_step=[],
        train_roll=[], val_roll=[],
    )

    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_rho(
            model=model,
            step_loader=step_train_loader,
            roll_loader=roll_train_loader,
            optimizer=optimizer,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
        )

        va, va_step, va_roll = run_epoch_eval_rho(
            model=model,
            val_step_loader=step_val_loader,
            val_roll_loader=roll_val_loader,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
        )

        scheduler.step(epoch - 1)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["lr"].append(lr)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 50 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:5d} | train={tr:.5f} val={va:.5f} "
                f"| step(tr/va)=({tr_step:.5f}/{va_step:.5f}) "
                f"| roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) "
                f"| best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop for RhoNet_v8 at epoch {epoch}  (best val={best_val:.5f} @ epoch {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    print(f"\nRestored best RhoNet_v8 model from epoch {history['best_epoch']}")
    torch.save(best_state, out_path)
    print(f"Saved RhoNet_v8 weights → {out_path}")

    return model, history, best_val


def evaluate_dp_model(model, X, Y_raw5, Y_std5, device):
    model.eval()
    with torch.no_grad():
        pred5_norm = model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

    pred5_phys = denormalize_Y_dp(pred5_norm, Y_std5)
    pred6_phys = reconstruct_Y6_from_Y5(pred5_phys)

    true6 = np.column_stack([
        Y_raw5[:, 0],
        Y_raw5[:, 1],
        -(Y_raw5[:, 0] + Y_raw5[:, 1]),
        Y_raw5[:, 2],
        Y_raw5[:, 3],
        Y_raw5[:, 4],
    ])

    r2 = np.zeros(6)
    rmse = np.zeros(6)

    for i in range(6):
        ss_res = np.sum((true6[:, i] - pred6_phys[:, i]) ** 2)
        ss_tot = np.sum((true6[:, i] - true6[:, i].mean()) ** 2)
        r2[i] = 1.0 - ss_res / (ss_tot + 1e-40)
        rmse[i] = np.sqrt(np.mean((true6[:, i] - pred6_phys[:, i]) ** 2))

    return r2, rmse, pred6_phys, true6


def evaluate_rho_model(model, X_rho_norm, Y_rho_raw_rate, Y_rho_std, device, seq_csv_path):

    model.eval()
    with torch.no_grad():
        pred_norm = model(torch.tensor(X_rho_norm, dtype=torch.float32).to(device)).cpu().numpy()

    pred_phys = denormalize_Y_rho(pred_norm, Y_rho_std)
    true = Y_rho_raw_rate.copy()

    ss_res = np.sum((true[:, 0] - pred_phys[:, 0]) ** 2)
    ss_tot = np.sum((true[:, 0] - true[:, 0].mean()) ** 2)
    r2 = 1.0 - ss_res / (ss_tot + 1e-40)
    rmse = np.sqrt(np.mean((true[:, 0] - pred_phys[:, 0]) ** 2))

    return r2, rmse, pred_phys, true


def predict_dp_once(dp_model, x_raw, EP_MAX, Y_dp_std):
    x_norm = normalize_X_dp(x_raw, EP_MAX)
    with torch.no_grad():
        pred5_norm = dp_model(torch.tensor(x_norm, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    pred5_phys = denormalize_Y_dp(pred5_norm, Y_dp_std)
    pred6_phys = reconstruct_Y6_from_Y5(pred5_phys)
    return pred6_phys[0]


def predict_rho_rate_once(rho_model, x_raw, EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    x_norm = normalize_X_rho(x_raw, EP_MAX, dt_log_mean, dt_log_std)
    with torch.no_grad():
        pred_norm = rho_model(torch.tensor(x_norm, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    pred_phys = denormalize_Y_rho(pred_norm, Y_rho_std)
    return float(pred_phys[0, 0])


def rollout_one_run_modeA(dp_model, run_df, EP_MAX, Y_dp_std):

    run_df = run_df.sort_values("k").reset_index(drop=True).copy()

    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)

    ep_pred = np.zeros(len(run_df), dtype=np.float64)
    ep_pred[0] = ep_true[0]

    rows = []

    for i in range(len(run_df)):
        row = run_df.iloc[i]

        x_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_true[i], row["loading"], ep_pred[i]
        ]], dtype=np.float64)

        pred6 = predict_dp_once(dp_model, x_raw, EP_MAX, Y_dp_std)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < len(run_df):
            dt = float(run_df.iloc[i]["dt"])
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * dt

    return pd.DataFrame(rows)


def rollout_one_run_modeB(dp_model, rho_model, run_df, EP_MAX, Y_dp_std, Y_rho_std, dt_log_mean, dt_log_std):

    run_df = run_df.sort_values("k").reset_index(drop=True).copy()

    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)
    dt_arr = run_df["dt"].values.astype(np.float64)

    ep_pred = np.zeros(len(run_df), dtype=np.float64)
    log10rho_pred = np.zeros(len(run_df), dtype=np.float64)

    ep_pred[0] = ep_true[0]
    log10rho_pred[0] = log10rho_true[0]

    rows = []

    for i in range(len(run_df)):
        row = run_df.iloc[i]

        x_dp_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_pred[i], row["loading"], ep_pred[i]
        ]], dtype=np.float64)

        pred6 = predict_dp_once(dp_model, x_dp_raw, EP_MAX, Y_dp_std)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        x_rho_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_pred[i], row["loading"], ep_pred[i], row["dt"]
        ]], dtype=np.float64)

        rho_rate_pred = predict_rho_rate_once(
            rho_model, x_rho_raw, EP_MAX, Y_rho_std, dt_log_mean, dt_log_std
        )

        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "log10rho_pred": float(log10rho_pred[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "rho_pred": float(10.0 ** log10rho_pred[i]),
            "dlog10rho_true": float(row["dlog10rho_true"]),
            "dlog10rho_rate_true": float(row["dlog10rho_rate_true"]),
            "rho_rate_pred": float(rho_rate_pred),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < len(run_df):
            dt = dt_arr[i]
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * dt
            log10rho_pred[i + 1] = log10rho_pred[i] + rho_rate_pred * dt

    return pd.DataFrame(rows)


def rollout_all_test_runs_modeA(dp_model, seq_csv_path, EP_MAX, Y_dp_std, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())

    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeA(dp_model, run_df, EP_MAX, Y_dp_std)

        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )

        all_summary.append({
            "stem": stem,
            "ep_eq_rel_err": ep_rel_err
        })

        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeA_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_path = os.path.join(out_dir, "rollout_summary_modeA.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nMode A rollout summary (pred ep_eq + TRUE density):")
    print(summary_df)
    print("\nMean rollout ep_eq error:")
    print(summary_df["ep_eq_rel_err"].mean())

    return summary_df


def rollout_all_test_runs_modeB(dp_model, rho_model, seq_csv_path, EP_MAX, Y_dp_std, Y_rho_std,
                                dt_log_mean, dt_log_std, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())

    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeB(
            dp_model, rho_model, run_df, EP_MAX, Y_dp_std, Y_rho_std, dt_log_mean, dt_log_std
        )

        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )

        rho_rel_err = np.mean(
            np.abs(pred_df["rho_pred"] - pred_df["rho_true"]) /
            np.maximum(np.abs(pred_df["rho_true"]), 1e-30)
        )

        all_summary.append({
            "stem": stem,
            "ep_eq_rel_err": ep_rel_err,
            "rho_rel_err": rho_rel_err
        })

        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeB_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_path = os.path.join(out_dir, "rollout_summary_modeB.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nMode B rollout summary (pred ep_eq + PRED density, rho-rate integration):")
    print(summary_df)
    print("\nMean rollout ep_eq error:")
    print(summary_df["ep_eq_rel_err"].mean())
    print("\nMean rollout rho error:")
    print(summary_df["rho_rel_err"].mean())

    return summary_df


def plot_training_curves(history, out_path_prefix):
    epochs = range(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history["train"], label="train")
    axes[0].plot(epochs, history["val"], label="val")
    axes[0].axvline(history["best_epoch"], color="red", ls=":", label=f"best={history['best_epoch']}")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss curves")
    axes[0].legend()

    axes[1].plot(epochs, history["lr"])
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("LR")
    axes[1].set_title("Cosine annealing LR with warm restarts")

    plt.tight_layout()
    plt.savefig(out_path_prefix + "_train_curves.png", dpi=150)
    plt.close()


def plot_branch_training_detail(history, out_path, title):
    epochs = range(1, len(history["train"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.plot(epochs, history["train_step"], label="train step")
    plt.plot(epochs, history["val_step"], label="val step")
    plt.plot(epochs, history["train_roll"], label="train roll")
    plt.plot(epochs, history["val_roll"], label="val roll")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_dp_per_component(r2s, rmses, out_dir):
    x = np.arange(6)
    w = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for j, split in enumerate(["train", "val", "test"]):
        axes[0].bar(x + (j - 1) * w, r2s[split], w, label=split)
        axes[1].bar(x + (j - 1) * w, rmses[split], w, label=split)

    axes[0].axhline(1.0, color="k", ls="--", lw=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[0].set_ylabel("R²")
    axes[0].set_title("Dp model: R² per component")
    axes[0].legend()

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("Dp model: RMSE per component")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "dp_per_component_metrics.png"), dpi=150)
    plt.close()


def plot_rollout_stress_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob

    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]

    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return

    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])

        if "uniaxial" in stem.lower():
            stress = df["s_zz"].values / 1e6
            ylabel = r"$\sigma_{zz}$ (MPa)"
        else:
            stress = df["s_xy"].values / 1e6
            ylabel = r"$\tau_{xy}$ (MPa)"

        ep_true = df["ep_eq_true"].values * 100.0
        ep_pred = df["ep_eq_pred"].values * 100.0

        ax.plot(ep_true, stress, label="DDD true path", linewidth=2)
        ax.plot(ep_pred, stress, "--", label="NN rollout path", linewidth=2)

        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


def plot_rollout_density_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob

    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]

    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return

    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])

        ep_true = df["ep_eq_true"].values * 100.0
        ax.plot(ep_true, df["rho_true"].values, label="DDD true density", linewidth=2)
        ax.plot(ep_true, df["rho_pred"].values, "--", label="NN rollout density", linewidth=2)

        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(r"$\rho$ (m$^{-2}$)")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


def train_and_evaluate_v8(npz_path, stats_path, seq_path):
    print(f"Device : {DEVICE}")
    print(f"PyTorch: {torch.__version__}\n")

    print("Loading dataset v8_cellsplit ...")
    data = np.load(npz_path)
    stats = json.load(open(stats_path))

    X_train_dp_raw = data["X_train_dp_raw"]
    X_val_dp_raw   = data["X_val_dp_raw"]
    X_test_dp_raw  = data["X_test_dp_raw"]

    Y_train_dp_raw = data["Y_train_dp_raw"]
    Y_val_dp_raw   = data["Y_val_dp_raw"]
    Y_test_dp_raw  = data["Y_test_dp_raw"]

    X_train_rho_raw = data["X_train_rho_raw"]
    X_val_rho_raw   = data["X_val_rho_raw"]
    X_test_rho_raw  = data["X_test_rho_raw"]

    Y_train_rho_raw = data["Y_train_rho_raw"]
    Y_val_rho_raw   = data["Y_val_rho_raw"]
    Y_test_rho_raw  = data["Y_test_rho_raw"]

    EP_MAX      = float(stats["EP_MAX"])
    Y_dp_std    = np.array(stats["Y_dp_std"], dtype=np.float64)
    Y_rho_std   = float(stats["Y_rho_std"])
    DT_LOG_MEAN = float(stats["DT_LOG_MEAN"])
    DT_LOG_STD  = float(stats["DT_LOG_STD"])

    print(f"  Train Dp: {X_train_dp_raw.shape[0]:,} | Val Dp: {X_val_dp_raw.shape[0]:,} | Test Dp: {X_test_dp_raw.shape[0]:,}")
    print(f"  Train Rho: {X_train_rho_raw.shape[0]:,} | Val Rho: {X_val_rho_raw.shape[0]:,} | Test Rho: {X_test_rho_raw.shape[0]:,}")
    print(f"  EP_MAX: {EP_MAX:.4e}")
    print(f"  Y_dp_std: {[f'{v:.1f}' for v in Y_dp_std]}")
    print(f"  Y_rho_std: {Y_rho_std:.4e}")
    print(f"  DT_LOG_MEAN: {DT_LOG_MEAN:.4f}  DT_LOG_STD: {DT_LOG_STD:.4f}")

    X_train_dp = normalize_X_dp(X_train_dp_raw, EP_MAX)
    X_val_dp   = normalize_X_dp(X_val_dp_raw,   EP_MAX)
    X_test_dp  = normalize_X_dp(X_test_dp_raw,  EP_MAX)

    Y_train_dp = normalize_Y_dp(Y_train_dp_raw, Y_dp_std)
    Y_val_dp   = normalize_Y_dp(Y_val_dp_raw,   Y_dp_std)
    Y_test_dp  = normalize_Y_dp(Y_test_dp_raw,  Y_dp_std)

    X_train_rho = normalize_X_rho(X_train_rho_raw, EP_MAX, DT_LOG_MEAN, DT_LOG_STD)
    X_val_rho   = normalize_X_rho(X_val_rho_raw,   EP_MAX, DT_LOG_MEAN, DT_LOG_STD)
    X_test_rho  = normalize_X_rho(X_test_rho_raw,  EP_MAX, DT_LOG_MEAN, DT_LOG_STD)

    Y_train_rho = normalize_Y_rho(Y_train_rho_raw, Y_rho_std)
    Y_val_rho   = normalize_Y_rho(Y_val_rho_raw,   Y_rho_std)
    Y_test_rho  = normalize_Y_rho(Y_test_rho_raw,  Y_rho_std)

    print("\nNormalized input ranges (train) — Dp branch:")
    for i, label in enumerate(INPUT_LABELS_DP):
        print(f"  {label:10s}: [{X_train_dp[:,i].min():.3f}, {X_train_dp[:,i].max():.3f}]  mean={X_train_dp[:,i].mean():.3f}")

    print("\nNormalized input ranges (train) — Rho branch:")
    for i, label in enumerate(INPUT_LABELS_RHO):
        print(f"  {label:10s}: [{X_train_rho[:,i].min():.3f}, {X_train_rho[:,i].max():.3f}]  mean={X_train_rho[:,i].mean():.3f}")

    dp_train_ds = NoisyDataset(X_train_dp, Y_train_dp, noise_std=INPUT_NOISE_STD, no_noise_indices=[8])
    dp_train_loader = DataLoader(
        dp_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda")
    )
    dp_val_loader = make_val_loader(X_val_dp, Y_val_dp, BATCH_SIZE)

    dp_roll_train_ds = DpRolloutWindowDataset(
        seq_csv_path=seq_path, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )
    dp_roll_val_ds = DpRolloutWindowDataset(
        seq_csv_path=seq_path, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )

    dp_roll_train_loader = DataLoader(
        dp_roll_train_ds, batch_size=ROLL_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )
    dp_roll_val_loader = DataLoader(
        dp_roll_val_ds, batch_size=ROLL_BATCH_SIZE, shuffle=False, num_workers=0
    )

    rho_train_ds = NoisyDataset(X_train_rho, Y_train_rho, noise_std=INPUT_NOISE_STD, no_noise_indices=[8, 10])
    rho_train_loader = DataLoader(
        rho_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda")
    )
    rho_val_loader = make_val_loader(X_val_rho, Y_val_rho, BATCH_SIZE)

    rho_roll_train_ds = RhoRolloutWindowDataset(
        seq_csv_path=seq_path, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )
    rho_roll_val_ds = RhoRolloutWindowDataset(
        seq_csv_path=seq_path, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )

    rho_roll_train_loader = DataLoader(
        rho_roll_train_ds, batch_size=ROLL_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )
    rho_roll_val_loader = DataLoader(
        rho_roll_val_ds, batch_size=ROLL_BATCH_SIZE, shuffle=False, num_workers=0
    )

    print(f"\nDp rollout windows: train={len(dp_roll_train_ds):,}, val={len(dp_roll_val_ds):,}")
    print(f"Rho rollout windows: train={len(rho_roll_train_ds):,}, val={len(rho_roll_val_ds):,}")

    dp_model = MLPRegressor(
        in_dim=10,
        out_dim=5,
        hidden_dims=DP_HIDDEN_DIMS,
        activation=DP_ACTIVATION,
        dropout=DP_DROPOUT,
        use_bn=DP_USE_BATCHNORM,
    ).to(DEVICE)

    n_dp_params = sum(p.numel() for p in dp_model.parameters() if p.requires_grad)
    print(f"\nDp model: {n_dp_params:,} trainable parameters")
    print(dp_model)

    dp_weights_path = os.path.join(NN_OUT_DIR, "dpnet_v8_best_state.pt")
    dp_model, dp_history, dp_best_val = train_dp_model(
        model=dp_model,
        step_train_loader=dp_train_loader,
        step_val_loader=dp_val_loader,
        roll_train_loader=dp_roll_train_loader,
        roll_val_loader=dp_roll_val_loader,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        out_path=dp_weights_path,
    )

    torch.save({
        "model_state_dict": dp_model.state_dict(),
        "model_config": {
            "in_dim": 10,
            "out_dim": 5,
            "hidden_dims": DP_HIDDEN_DIMS,
            "activation": DP_ACTIVATION,
            "dropout": DP_DROPOUT,
            "use_bn": DP_USE_BATCHNORM,
        },
        "norm_config": {
            "STRESS_SCALE": STRESS_SCALE,
            "LOGRHO_SHIFT": LOGRHO_SHIFT,
            "SDOT_LOG_SHIFT": SDOT_LOG_SHIFT,
            "EP_MAX": EP_MAX,
            "Y_dp_std": Y_dp_std.tolist(),
        },
        "training_config": {
            "window_len": WINDOW_LEN,
            "window_stride": WINDOW_STRIDE,
            "roll_start_epoch": ROLL_START_EPOCH,
            "W_STEP_DP": W_STEP_DP,
            "W_ROLL_DP": W_ROLL_DP,
        },
        "best_val_loss": dp_best_val,
        "input_order": INPUT_LABELS_DP,
        "output_note": "Predicts 5 free Dp with rollout-window training. Dp_zz = -(Dp_xx+Dp_yy). Trained with cell_grouped_split_v8.",
    }, os.path.join(NN_OUT_DIR, "dpnet_v8_best.pt"))

    rho_model = MLPRegressor(
        in_dim=11,
        out_dim=1,
        hidden_dims=RHO_HIDDEN_DIMS,
        activation=RHO_ACTIVATION,
        dropout=RHO_DROPOUT,
        use_bn=RHO_USE_BATCHNORM,
    ).to(DEVICE)

    n_rho_params = sum(p.numel() for p in rho_model.parameters() if p.requires_grad)
    print(f"\nRho model: {n_rho_params:,} trainable parameters")
    print(rho_model)

    rho_weights_path = os.path.join(NN_OUT_DIR, "rhonet_v8_best_state.pt")
    rho_model, rho_history, rho_best_val = train_rho_model(
        model=rho_model,
        step_train_loader=rho_train_loader,
        step_val_loader=rho_val_loader,
        roll_train_loader=rho_roll_train_loader,
        roll_val_loader=rho_roll_val_loader,
        EP_MAX=EP_MAX,
        Y_rho_std=Y_rho_std,
        dt_log_mean=DT_LOG_MEAN,
        dt_log_std=DT_LOG_STD,
        out_path=rho_weights_path,
    )

    torch.save({
        "model_state_dict": rho_model.state_dict(),
        "model_config": {
            "in_dim": 11,
            "out_dim": 1,
            "hidden_dims": RHO_HIDDEN_DIMS,
            "activation": RHO_ACTIVATION,
            "dropout": RHO_DROPOUT,
            "use_bn": RHO_USE_BATCHNORM,
        },
        "norm_config": {
            "STRESS_SCALE": STRESS_SCALE,
            "LOGRHO_SHIFT": LOGRHO_SHIFT,
            "SDOT_LOG_SHIFT": SDOT_LOG_SHIFT,
            "EP_MAX": EP_MAX,
            "Y_rho_std": Y_rho_std,
            "DT_LOG_MEAN": DT_LOG_MEAN,
            "DT_LOG_STD": DT_LOG_STD,
        },
        "training_config": {
            "window_len": WINDOW_LEN,
            "window_stride": WINDOW_STRIDE,
            "roll_start_epoch": ROLL_START_EPOCH,
            "W_STEP_RHO": W_STEP_RHO,
            "W_ROLL_RHO": W_ROLL_RHO,
        },
        "best_val_loss": rho_best_val,
        "input_order": INPUT_LABELS_RHO,
        "output_note": "Predicts d(log10rho)/dt with rollout-window training on log10rho trajectory. Trained with cell_grouped_split_v8.",
    }, os.path.join(NN_OUT_DIR, "rhonet_v8_best.pt"))

    print("\nEvaluating Dp model on all splits ...")
    dp_r2s, dp_rmses = {}, {}
    dp_preds, dp_trues = {}, {}
    for split, X, Y_raw in [
        ("train", X_train_dp, Y_train_dp_raw),
        ("val",   X_val_dp,   Y_val_dp_raw),
        ("test",  X_test_dp,  Y_test_dp_raw),
    ]:
        r2, rmse, pred6, true6 = evaluate_dp_model(dp_model, X, Y_raw, Y_dp_std, DEVICE)
        dp_r2s[split] = r2
        dp_rmses[split] = rmse
        dp_preds[split] = pred6
        dp_trues[split] = true6

    print(f"\n{'Component':<12}{'R²_train':>10}{'R²_val':>10}{'R²_test':>10}{'RMSE_test':>12}")
    print("-" * 56)
    for i, label in enumerate(DP_OUTPUT_LABELS_6):
        note = " *" if label == "Dp_zz" else "  "
        print(f"  {label:<8}{note}  {dp_r2s['train'][i]:>8.4f}  {dp_r2s['val'][i]:>8.4f}  {dp_r2s['test'][i]:>8.4f}  {dp_rmses['test'][i]:>10.2f}")
    print("-" * 56)
    print(f"  {'MEAN':<10}  {dp_r2s['train'].mean():>8.4f}  {dp_r2s['val'].mean():>8.4f}  {dp_r2s['test'].mean():>8.4f}  {dp_rmses['test'].mean():>10.2f}")

    print("\nEvaluating improved rho model on all splits ...")
    rho_metrics = {}
    for split, X, Y_raw in [
        ("train", X_train_rho, Y_train_rho_raw),
        ("val",   X_val_rho,   Y_val_rho_raw),
        ("test",  X_test_rho,  Y_test_rho_raw),
    ]:
        r2, rmse, pred, true = evaluate_rho_model(rho_model, X, Y_raw, Y_rho_std, DEVICE, seq_path)
        rho_metrics[split] = dict(r2=r2, rmse=rmse)

    print(f"\n{'Split':<10}{'R²':>12}{'RMSE':>16}")
    print("-" * 38)
    for split in ["train", "val", "test"]:
        print(f"{split:<10}{rho_metrics[split]['r2']:>12.4f}{rho_metrics[split]['rmse']:>16.4e}")

    print("\nGenerating plots ...")
    plot_training_curves(dp_history, os.path.join(NN_OUT_DIR, "dp"))
    plot_training_curves(rho_history, os.path.join(NN_OUT_DIR, "rho"))
    plot_branch_training_detail(dp_history, os.path.join(NN_OUT_DIR, "dp_rollout_training_detail.png"),
                                "DpNet step vs rollout losses")
    plot_branch_training_detail(rho_history, os.path.join(NN_OUT_DIR, "rho_rollout_training_detail.png"),
                                "RhoNet step vs rollout losses")
    plot_dp_per_component(dp_r2s, dp_rmses, NN_OUT_DIR)

    rollout_summary_A = rollout_all_test_runs_modeA(
        dp_model=dp_model,
        seq_csv_path=seq_path,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        out_dir=NN_OUT_DIR,
    )

    rollout_summary_B = rollout_all_test_runs_modeB(
        dp_model=dp_model,
        rho_model=rho_model,
        seq_csv_path=seq_path,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        Y_rho_std=Y_rho_std,
        dt_log_mean=DT_LOG_MEAN,
        dt_log_std=DT_LOG_STD,
        out_dir=NN_OUT_DIR,
    )

    print("\nDisplaying rollout Mode A: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeA", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: density vs plastic strain ...")
    plot_rollout_density_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print(f"\nAll outputs in: {NN_OUT_DIR}")
    print("  dpnet_v8_best.pt")
    print("  rhonet_v8_best.pt")
    print("  dp_train_curves.png | rho_train_curves.png")
    print("  dp_rollout_training_detail.png | rho_rollout_training_detail.png")
    print("  dp_per_component_metrics.png")
    print("  rollout_summary_modeA.csv")
    print("  rollout_summary_modeB.csv")
    print("  rollout_modeA_<runname>.csv")
    print("  rollout_modeB_<runname>.csv")

    return (
        dp_model, rho_model,
        dp_history, rho_history,
        dp_r2s, rho_metrics,
        rollout_summary_A, rollout_summary_B
    )

if __name__ == "__main__":
    set_all_seeds(RANDOM_SEED)

    dataset_paths = build_dataset_v8()

    (
        dp_model, rho_model,
        dp_history, rho_history,
        dp_r2s, rho_metrics,
        rollout_summary_A, rollout_summary_B
    ) = train_and_evaluate_v8(
        npz_path=dataset_paths["npz_path"],
        stats_path=dataset_paths["stats_path"],
        seq_path=dataset_paths["seq_path"],
    )

In [ ]:

# Uniaxial Stress Controlled Loading


import os, sys, csv          #IMporting basic python libraries
import numpy as np

# import pyexadis
pyexadis_paths = [
    '../../python',
    '../../lib',
    '../../core/pydis/python',        # possible paths where OpenDiS Python modules may exist.
    '../../core/exadis/python/'
]

[
    sys.path.append(os.path.abspath(p))
    for p in pyexadis_paths
    if p not in sys.path
]

import pyexadis        # importing main pyexadis module
from pyexadis_base import ExaDisNet, DisNetManager
from pyexadis_base import CalForce, MobilityLaw, TimeIntegration
from pyexadis_base import Collision, Topology, Remesh, SimulateNetwork


#  helper functions

def voigt_sym_to_tensor(v6):
    #  converts Voigt order: [xx, yy, zz, yz, zx, xy] into 3×3 symmetric tensor.
    v6 = np.asarray(v6, float).ravel()

    return np.array([
        [v6[0], v6[5], v6[4]],
        [v6[5], v6[1], v6[3]],
        [v6[4], v6[3], v6[2]]
    ], float)


def von_mises_stress(S):                     # Calculating von Mises stress from a 3×3 stress tensor.
    S = np.array(S, float).reshape(3, 3)
    Sdev = S - np.trace(S)/3.0*np.eye(3)

    return np.sqrt(1.5*np.tensordot(Sdev, Sdev, 2))


def von_mises_eq_strain(E):                  # Calculating equivalent strain
    E = np.array(E, float).reshape(3, 3)
    Edev = E - np.trace(E)/3.0*np.eye(3)

    return np.sqrt((2.0/3.0)*np.tensordot(Edev, Edev, 2))


def Eeq(LA, MU):                              # Calculating Young’s modulus E from Lambda & shear modulus
    return MU*(3*LA + 2*MU)/(LA + MU)


def nu(LA, MU):
    return 0.5*LA/(LA + MU)                   # Calculating Poissons Ratio


class StopSimulation(RuntimeError):
    # Used to stop the simulation once we reach taregt stress and not max_step
    pass


class MySimulateNetworkPerf(SimulateNetwork):  # We are using default OpenDiS simulationdriver but customizing it

    def __init__(
        self,
        *args,
        LA,
        MU,
        stress_rate_tensor=None,              # Initializing the simulation object
        # stop controls
        t_total=None,
        target_sigma_zz=None,
        tol_time=0.0,
        tol_sigma=0.0,
        **kwargs
    ):

        super().__init__(*args, **kwargs)

        self.LA = float(LA)                    # Storing lambda and mu
        self.MU = float(MU)

        self._Sdot_voigt = np.asarray(
            stress_rate_tensor
            if stress_rate_tensor is not None    # storing stress rate tensor in Voigt format.
            else np.zeros(6),
            float
        )

        self.t_total = None if t_total is None else float(t_total)  # Storing total simulation time
        self.target_sigma_zz = (
            None if target_sigma_zz is None
            else float(target_sigma_zz)             # Storing the target axial stress.
        )

        self.tol_time = float(tol_time)
        self.tol_sigma = float(tol_sigma)          # tolerance values so that the simulation can terminate as it is very close to target

        # robust user-time accumulator
        self._time_cum = 0.0

        # output paths and storing the output folders
        self._out_dir = self.write_dir

        self._out_path = os.path.join(
            self._out_dir,
            "stress_strain_dens.dat"                # Creates path for output files
        )

        self._csv_main = os.path.join(
            self._out_dir,
            "stress_strain_dens_main.csv"
        )                                           # Here we have main csv and a power csv file where power file only has thermodynamic check

        self._csv_power = os.path.join(
            self._out_dir,
            "stress_strain_dens_power.csv"
        )

        self._wrote_header_dat = os.path.exists(self._out_path)
        self._wrote_header_csv = os.path.exists(self._csv_main)
        self._wrote_header_pow = os.path.exists(self._csv_power)

        self._step_fallback = 0
        self._ep_eq_cum = 0.0     # stores accumulated equivalent plastic strain:

    def step_update_response(self, N: DisNetManager, state: dict):      # Main update function which we call every step
                                                                        #  N is dislocation network manager.
        if self.loading_mode == 'stress_rate_tensor':

            dEp_T, dWp_vec, density = (
                N.get_disnet(ExaDisNet)
                 .net
                 .get_plastic_strain()
            )

            state["density"] = float(density)                           # Storing Density

            # extracting plastic deformation from the dislocation network.
            dEp = np.array(dEp_T, float).ravel()[[0, 4, 8, 5, 2, 1]]
                                                                           # The ordering is as per the convention in library
            dWp = np.array(dWp_vec, float).ravel()[[5, 2, 1]]
            # (Plastic Spin)
            state["dEp"] = dEp
            state["dWp"] = dWp          # plastic strain increment and plastic spin increment saved

            dt = float(state["dt"])

            # clamping final increment
            if self.t_total is not None:

                rem = self.t_total - self._time_cum

                if rem <= 0.0:
                    dt_eff = 0.0
                else:
                    dt_eff = min(dt, rem)

            else:
                dt_eff = dt

            # advance our own time
            self._time_cum += dt_eff

            # apply stress increment
            dS_voigt = self._Sdot_voigt * dt_eff      # Computing stress increment

            state["applied_stress"] = (
                state.get("applied_stress", np.zeros(6))    # Updating applied stress
                + dS_voigt
            )

            # compliance matrix formation
            E_ = Eeq(self.LA, self.MU)
            nu_ = nu(self.LA, self.MU)

            S = np.zeros((6, 6), float)

            for i in range(3):
                for j in range(3):
                    S[i, j] = -nu_/E_

                S[i, i] = 1.0/E_

            for i in range(3, 6):
                S[i, i] = 1.0/(2.0*self.MU)

            dEe = S @ dS_voigt          # Elastic Strain increment
            dE_voigt = dEe + dEp        # Total Strain increment


            state["Etot"] = (
                state.get("Etot", np.zeros(6))      # Accumulated total strain
                + dE_voigt
            )

            state["stress"] = float(
                von_mises_stress(
                    voigt_sym_to_tensor(state["applied_stress"])
                )
            )

            state["strain"] = float(
                von_mises_eq_strain(
                    voigt_sym_to_tensor(state["Etot"])
                )
            )

        else:

            super().step_update_response(N, state)

            self._time_cum += float(state.get("dt", 0.0))

        return state

    #function is called at the end of each simulation step.
    def step_end(self, N: DisNetManager, state: dict):

        super().step_end(N, state)

        t = float(self._time_cum)

        sig = np.asarray(
            state.get("applied_stress", np.zeros(6)),
            float
        ).ravel()

        sigma_zz = float(sig[2]) if sig.size >= 3 else 0.0

        time_hit = (
            (self.t_total is not None)
            and
            (t >= self.t_total - self.tol_time)
        )

        sigma_hit = (
            (self.target_sigma_zz is not None)
            and
            (sigma_zz >= self.target_sigma_zz - self.tol_sigma)
        )

        if time_hit or sigma_hit:

            raise StopSimulation(
                f"Stop: time={t:.6e}s, sigma_zz={sigma_zz:.6e}Pa"
            )
    # Output WRiting Function
    def step_write_files(self, N: DisNetManager, state: dict):

        step = int(
            state.get(
                "istep",
                state.get("step", self._step_fallback + 1)
            )
        )

        if self.write_freq and (                    # Controlling the output frequency
            step % int(self.write_freq) != 0
        ):
            return

        os.makedirs(self._out_dir, exist_ok=True)

        if not self._wrote_header_dat:

            with open(self._out_path, "w") as f:

                f.write(
                    "# step time(s) dt(s) strain_eq sigma_vm(Pa) "
                    "s_xx(Pa) s_yy(Pa) s_zz(Pa) "
                    "s_yz(Pa) s_zx(Pa) s_xy(Pa) "
                    "density(1/m^2) "                        # File headers
                    "Lpxx Lpxy Lpxz Lpyx Lpyy "
                    "Lpyz Lpzx Lpzy Lpzz "
                    "epdot_eq(1/s) ep_eq(-)\n"
                )

            self._wrote_header_dat = True

        if not self._wrote_header_csv:

            with open(self._csv_main, "w", newline="") as fcsv:

                w = csv.writer(fcsv)

                w.writerow([
                    "step",
                    "time(s)",
                    "dt(s)",
                    "strain_eq",
                    "sigma_vm(Pa)",
                    "s_xx(Pa)",
                    "s_yy(Pa)",
                    "s_zz(Pa)",
                    "s_yz(Pa)",
                    "s_zx(Pa)",
                    "s_xy(Pa)",
                    "density(1/m^2)",
                    "Lpxx",
                    "Lpxy",
                    "Lpxz",
                    "Lpyx",
                    "Lpyy",
                    "Lpyz",
                    "Lpzx",
                    "Lpzy",
                    "Lpzz",
                    "epdot_eq(1/s)",
                    "ep_eq(-)"
                ])

            self._wrote_header_csv = True

        if not self._wrote_header_pow:

            with open(self._csv_power, "w", newline="") as fcsv:

                w = csv.writer(fcsv)

                w.writerow([
                    "step",
                    "p_plast(W/m^3)",
                    "p_spin(W/m^3)",
                    "p_full(W/m^3)"
                ])

            self._wrote_header_pow = True

        dEp = voigt_sym_to_tensor(state["dEp"])  # converting plastic strain increment from Voigt to tensor:

        wyz, wxz, wxy = state["dWp"]

        dWp = np.array([
            [0.0,  wxy,  wxz],
            [-wxy, 0.0,  wyz],
            [-wxz, -wyz, 0.0]
        ], float)

        dt = float(state["dt"])
        time = float(self._time_cum)

        Lp = (dEp + dWp) / max(dt, 1e-30)   # Constructing Lp and Dp and 1e-40 is used for avoiding division by zero, a code choice
        Dp = dEp / max(dt, 1e-30)

        epdot_eq = float(
            np.sqrt(
                (2.0/3.0)
                *
                np.tensordot(Dp, Dp, axes=2)
            )
        )

        self._ep_eq_cum += epdot_eq * dt

        sigT = voigt_sym_to_tensor(state["applied_stress"])

        p_plast = float(np.tensordot(sigT, Dp, axes=2))

        p_spin = float(
            np.tensordot(
                sigT,
                dWp/max(dt, 1e-30),
                axes=2
            )
        )

        p_full = p_plast + p_spin

        strain_scalar = float(state.get("strain", 0.0))
        sigma_vm = float(state.get("stress", 0.0))
        density_scalar = float(state.get("density", 0.0))

        v6 = np.asarray(
            state["applied_stress"],
            float
        ).ravel()

        # Writing output rows at each step
        with open(self._out_path, "a") as f:

            f.write(
                f"{step:8d} "
                f"{time: .8e} "
                f"{dt: .8e} "
                f"{strain_scalar: .8e} "
                f"{sigma_vm: .8e} "
                +
                " ".join(f"{x: .8e}" for x in v6)
                +
                " "
                +
                f"{density_scalar: .8e} "
                +
                " ".join(
                    f"{x: .6e}"
                    for x in Lp.ravel(order='C')
                )
                +
                " "
                +
                f"{epdot_eq: .6e} "
                f"{self._ep_eq_cum: .6e}\n"
            )

            f.write(
                f" {p_plast: .6e} "
                f"{p_spin: .3e} "
                f"{p_full: .6e}\n"
            )

        with open(self._csv_main, "a", newline="") as fcsv:  # for main csv file

            w = csv.writer(fcsv)

            row = (
                [step, time, dt, strain_scalar, sigma_vm]
                +
                list(v6)
                +
                [density_scalar]
                +
                list(Lp.ravel(order='C'))
                +
                [epdot_eq, self._ep_eq_cum]
            )

            w.writerow(row)

        with open(self._csv_power, "a", newline="") as fcsv:  # For power csv file

            w = csv.writer(fcsv)

            w.writerow([
                step,
                p_plast,
                p_spin,
                p_full
            ])


def run_stress_controlled():       # This function sets up and runs the full DDD simulation.

    pyexadis.initialize()

    try:

        state = {
            "crystal": 'fcc',
            "burgmag": 2.55e-10,
            "mu": 54.6e9,
            "nu": 0.324,
            "a": 6.0,                          # Do not change until you see max_conn error and keep this constant
            "maxseg": 5000.0,
            "minseg": 1200.0,
            "rtol": 13.0,
            "rann": 9.0,
            "nextdt": 2e-11,
            "maxdt": 1.5e-10,
        }

        MU = float(state["mu"])
        nu_ = float(state["nu"])

        LA = 2.0*MU*nu_/(1.0 - 2.0*nu_)

        Lbox = 58824.0

        G = ExaDisNet().generate_line_config(
            state["crystal"],
            Lbox,
            num_lines=15*12,   # Change this for dislocation density
            theta=[0.0, 60.0, 90.0],
            maxseg=state["maxseg"],
            seed=1234
        )

        net = DisNetManager(G)

        calforce = CalForce(
            force_mode='SUBCYCLING_MODEL',
            state=state,                           # Force calculator
            Ngrid=64,
            cell=net.cell
        )

        mobility = MobilityLaw(
            mobility_law='FCC_0',
            state=state,                            # Velocity update
            Medge=64103.0,
            Mscrew=64103.0,
            vmax=2000.0
        )

        timeint = TimeIntegration(
            integrator='Subcycling',                 # Time integrator which updates position
            rgroups=[0.0, 80.0, 400.0, 1400.0],
            state=state,
            force=calforce,
            mobility=mobility
        )

        collision = Collision(
            collision_mode='Retroactive',
            state=state
        )

        topology = Topology(
            topology_mode='TopologyParallel',
            state=state,
            force=calforce,
            mobility=mobility
        )

        remesh = Remesh(
            remesh_rule='LengthBased',
            state=state
        )

        # RATE CONTROL INPUTS

        target_sigma = 40e6  # Change this for different sresses
        sigma_dot = 8e13     # Change this for rate change

        t_ramp = target_sigma / sigma_dot

        stress_rate_tensor = np.array(
            [0., 0., sigma_dot, 0., 0., 0.],
            float
        )

        write_dir = "40Mpa_8e13"      # Change this for every new simulation

        sim = MySimulateNetworkPerf(
            calforce=calforce,
            mobility=mobility,
            timeint=timeint,
            collision=collision,
            topology=topology,
            remesh=remesh,
            vis=None,
            loading_mode='stress_rate_tensor',
            burgmag=state["burgmag"],
            state=state,
            print_freq=100,
            plot_freq=0,
            write_freq=1,
            write_dir=write_dir,
            LA=LA,
            MU=MU,
            stress_rate_tensor=stress_rate_tensor,
            t_total=t_ramp,
            target_sigma_zz=target_sigma,
            tol_time=0.0,
            tol_sigma=0.0,
            max_step=100000   # JUst a cap
        )

        sim.run(net, state)

    except StopSimulation as e:

        print(str(e))

    finally:

        pyexadis.finalize()


if __name__ == "__main__":
    run_stress_controlled()